# Lab 1: ReAct 에이전트 기초

## ReAct = Reasoning + Acting
**생각하고(Thought) → 행동하고(Action) → 관찰하는(Observation) AI**

기존 LLM은 질문에 바로 답합니다. ReAct 에이전트는 다릅니다:
1. **Thought**: 무엇을 해야 할지 추론
2. **Action**: 도구(Tool)를 선택하고 실행
3. **Observation**: 결과를 확인
4. → 목표 달성까지 반복

**학습 목표:**
- ReAct 패턴의 Thought → Action → Observation 루프 이해
- 제조 현장 시나리오로 에이전트 동작 시뮬레이션
- LangSmith 연동 준비

In [ ]:
"""ReAct 에이전트 시뮬레이션 (LLM API 없이)"""

class ManufacturingReActAgent:
    """제조 현장 ReAct 에이전트"""
    
    def __init__(self, tools):
        self.tools = {t.__name__: t for t in tools}
        self.thought_log = []
    
    def run(self, task: str, max_steps=5):
        print(f"🎯 Task: {task}")
        print("=" * 50)
        
        for step in range(1, max_steps + 1):
            # Thought (추론)
            thought = self._think(task, step)
            print(f"💭 Thought [{step}]: {thought}")
            
            # Action (행동 결정)
            action, action_input = self._decide_action(task, step)
            print(f"⚡ Action [{step}]: {action}({action_input})")
            
            # Observation (결과 관찰)
            if action in self.tools:
                observation = self.tools[action](action_input)
            else:
                observation = "도구를 찾을 수 없습니다"
            print(f"👁️  Observation [{step}]: {observation}")
            print()
            
            self.thought_log.append({
                "step": step,
                "thought": thought,
                "action": action,
                "action_input": action_input,
                "observation": observation
            })
            
            # 종료 조건
            if "완료" in observation or "정상" in observation:
                print("✅ 목표 달성!")
                break
        
        return self.thought_log
    
    def _think(self, task, step):
        thoughts = {
            1: "먼저 현재 설비 상태를 확인해야 한다",
            2: "센서값이 임계치를 넘었다. 이상 여부를 판단해야 한다",
            3: "이상 감지됨. 정비팀에 알림을 보내야 한다",
            4: "알림을 보냈다. 작업 지시서를 발행해야 한다",
            5: "모든 조치를 취했다. 최종 보고를 남긴다",
        }
        return thoughts.get(step, "상황을 종합적으로 판단 중...")
    
    def _decide_action(self, task, step):
        actions = {
            1: ("check_sensor", "line_A_bearing"),
            2: ("detect_anomaly", "vibration=4.2mm/s"),
            3: ("send_alert", "베어링 이상 감지 - 즉시 점검 필요"),
            4: ("create_work_order", "라인 A 베어링 교체"),
            5: ("report", "완료"),
        }
        return actions.get(step, ("report", "완료"))


# ===== 도구(Tool) 정의 =====

def check_sensor(sensor_id: str) -> str:
    """설비 센서 값 조회"""
    sensors = {
        "line_A_bearing": "진동: 4.2mm/s (임계값: 3.0) ⚠️ 초과",
        "line_B_temp": "온도: 68°C (임계값: 80°C) ✅ 정상",
        "line_C_pressure": "압력: 5.1 bar (임계값: 6.0 bar) ✅ 정상",
    }
    return sensors.get(sensor_id, f"'{sensor_id}' 센서 없음")

def detect_anomaly(reading: str) -> str:
    """센서 읽값의 이상 여부 판단"""
    try:
        val = float(reading.split("=")[1].replace("mm/s", "").strip())
        if val > 3.0:
            return f"⚠️ 이상 감지 (측정값: {val}mm/s, 임계값: 3.0mm/s 초과)"
        else:
            return f"정상 범위 (측정값: {val}mm/s)"
    except Exception:
        return f"판단 불가: {reading}"

def send_alert(message: str) -> str:
    """정비팀 및 관리자에게 알림 전송"""
    return f"📱 알림 전송 완료: {message} (수신: 정비팀장, 생산관리자)"

def create_work_order(task: str) -> str:
    """작업 지시서 생성"""
    import datetime
    order_id = f"WO-{datetime.datetime.now().strftime('%m%d%H%M')}"
    return f"작업 지시서 생성 완료: {order_id} | 작업: {task}"

def report(status: str) -> str:
    """최종 보고서 생성"""
    return f"📋 최종 보고 완료: {status} — 모든 조치 기록됨"


# ===== 에이전트 실행 =====
agent = ManufacturingReActAgent([
    check_sensor, detect_anomaly, send_alert, create_work_order, report
])

log = agent.run("라인 A 베어링 상태 모니터링 후 이상 시 알림 및 작업 지시")

In [ ]:
# LangSmith 연동 준비 코드 (API 키 설정 방법 안내)
import os

# LangSmith 설정 (API 키 있을 때만 활성화)
# os.environ["LANGSMITH_API_KEY"] = "your-key-here"
# os.environ["LANGSMITH_PROJECT"] = "manufacturing-ai-koreatech"
# os.environ["LANGCHAIN_TRACING_V2"] = "true"

print("LangSmith 연동 방법:")
print("1. https://smith.langchain.com 에서 API 키 발급")
print("2. .env 파일에 LANGSMITH_API_KEY=... 설정")
print("3. 위 주석 해제 후 실행")
print("4. LangSmith 대시보드에서 Thought/Action/Observation trace 확인")
print()
print("✅ 현재: mock 모드로 실행 중 (API 키 없이도 학습 가능)")
print()

# 실행 로그 요약 출력
print("=" * 50)
print("ReAct 실행 로그 요약")
print("=" * 50)
for entry in agent.thought_log:
    print(f"Step {entry['step']}:")
    print(f"  💭 {entry['thought']}")
    print(f"  ⚡ {entry['action']}({entry['action_input']})")
    print(f"  👁️  {entry['observation']}")

## ReAct 핵심 정리

**Thought → Action → Observation 루프**

```
사용자 Task
    ↓
[Thought]  : "먼저 센서를 확인해야 한다"
    ↓
[Action]   : check_sensor("line_A_bearing")
    ↓
[Observation]: "진동: 4.2mm/s ⚠️ 초과"
    ↓
[Thought]  : "이상 감지됨. 알림을 보내야 한다"
    ↓
[Action]   : send_alert("베어링 이상 감지")
    ↓
[Observation]: "📱 알림 전송 완료"
    ↓
  목표 달성!
```

**ReAct vs 기존 LLM 비교:**
| 기존 LLM | ReAct Agent |
|---|---|
| 질문 → 즉시 답변 | 생각 → 행동 → 관찰 → 반복 |
| 외부 시스템 접근 불가 | Tool로 MES, 센서 등 연동 |
| 단발성 답변 | 복합 목표 달성 가능 |

**다음 Lab:** 제조 현장 특화 Tool 4개를 구현합니다.